In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.4 Continuous batching

> 🎯 **Goal:** Serve several requests in a single forward pass over a padded, masked batch, and understand how a real server lets requests join and leave mid-flight.

**Why this matters.** A GPU is a massively parallel machine that sits half-idle if you feed it one sequence at a time. Throughput (tokens served per second across all users) collapses. Continuous batching is how production servers like vLLM and TGI keep the GPU busy and serve many users at once without making any single user wait in a long queue.

**The intuition.** Two words to pin down first. **Latency** is how long one user waits for their answer. **Throughput** is how many users you can serve per second in total. They pull against each other, and continuous batching is how you raise throughput without wrecking latency. Picture a busy kitchen. A naive batch is a fixed tray: you wait until the tray is full, cook everything together, and nobody gets served until the slowest dish is done. Continuous batching is a good line cook who keeps many orders going at once and lets a new order join the moment a finished one leaves the pan, instead of waiting for a whole fresh tray.

**The mechanics.** The core move is a single forward pass over many sequences at once. Because prompts have different lengths, you **pad** the short ones to a common length and carry a **mask** marking which positions are real. The model runs once over the whole padded batch; then for each prompt you pick out the logits at its own last *real* token (using the mask) to get that prompt's next-token prediction. A production server wraps this in a loop: every step it admits new requests into the running batch and retires finished ones, so the batch is continuously refilled rather than drained and restarted. The code below shows the heart of it, one forward serving three different-length prompts.

In [ ]:
from pocketlm import train_tiny, pick_config, pad_batch

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
prompts = ["The ", "ROMEO", "a"]
ids = [tok.encode(p) for p in prompts]
padded, mask = pad_batch(ids, pad_id=0)
logits, _ = model(padded)
last = mask.sum(1) - 1                                  # each prompt's final real token
nxt = logits[torch.arange(len(prompts)), last].argmax(-1)
for p, i in zip(prompts, nxt):
    print(f"{p!r:>8} -> next char {tok.itos[int(i)]!r}")

**What just happened.** Three prompts of different lengths went through the model in one forward pass, and for each we read the next-character prediction at its own final real token. That single batched call is the unit of work a real server schedules. The server keeps this batch *running*: every step it admits new requests and retires finished ones, keeping GPU utilization high (good throughput) while no single request waits for a full tray to fill (good latency).

⚠️ **Common pitfalls**
- Reading logits at the padded last column instead of the last *real* token. The mask exists precisely so you index the right position per row.
- Letting padding tokens attend or contribute to the loss. Padding is filler, it must be masked out everywhere.
- Padding everything to the global max length, which wastes compute. Real schedulers group similar lengths and pad to the batch max, not the model max.

🏋️ **Try it yourself.** Add a fourth, much longer prompt to the list and rerun. Confirm each prompt still gets the correct next char by checking that the per-row `last` index lands on its final real token. Then print `padded.shape` to see how the batch padded up to the longest prompt.

In [ ]:
assert tuple(nxt.shape) == (len(prompts),)